# 0 - Construction de la base de données

Ce notebook présente les fonctionnalités du module `builders` qui contient un ensemble de classes permettant différentes configurations de création de bases de données DuckLake.

### Table des matières

0. [Importation des modules](#section_0)
1. [Création des données synthétiques](#section_1)
2. [Construction du schéma](#section_2)
   - [Initialisation de la classe](#section_2_1)
   - [Construction des méta-données](#section_2_2)
   - [Construction des tables de dimensions](#section_2_3)
   - [Construction de la table d'information](#section_2_4)
   - [Création de l'ensemble des tables](#section_2_5)
   - [Compatibilité multi-backend — pandas, polars et narwhals](#section_2_6)
3. [Construction de la base de données (DuckLake)](#section_3)
   - [Construction d'une base de données sans clé primaire](#section_3_1)
     * [Connexion DuckLake et initialisation du builder](#section_3_1_1)
     * [Construction du schéma](#section_3_1_2)
     * [Affichage du schéma](#section_3_1_3)
     * [Exemple de requête](#section_3_1_4)
   - [Construction d'une base de données avec clé primaire](#section_3_2)
     * [Création d'une clé primaire composite](#section_3_2_1)
     * [Condition d'unicité sur la clé primaire lors de la création de la table](#section_3_2_2)
4. [categorical_threshold=None — pas de tables de dimension](#section_4)

## 0. Importation des modules <a id="section_0"></a>

In [ ]:
# Rechargement automatique des modules
%load_ext autoreload
%autoreload 2

# Modules de base
import os
import shutil
import sys
from datetime import datetime, timedelta

import narwhals as nw
import numpy as np
import pandas as pd
import polars as pl

# Ajout du chemin
sys.path.append("..")

# Importation des modules ad hoc
from dt_ducklake_manager.connection import DuckLakeConnector
from dt_ducklake_manager.schema import DuckLakeTablesBuilder, SchemaBuilder

## 1. Création des données synthétiques <a id="section_1"></a>

In [ ]:
# Définition des paramètres pour les données synthétiques
np.random.seed(42)  # Pour la reproductibilité

# Paramètres
n_rows = 500
start_date = datetime(2023, 1, 1)

# Création des colonnes catégorielles
indicators = ["temperature", "humidity", "pressure", "wind_speed"]
countries = ["France", "Germany", "Italy", "Spain", "Belgium"]
kinds = ["forecast", "observation"]
models = ["model_A", "model_B", "model_C"]
trainings = ["train_v1", "train_v2"]
weeks = list(range(1, 53))
horizons = [1, 3, 7, 14, 30]

# Génération de données uniques pour la clé primaire composite
# Création de toutes les combinaisons possibles puis échantillonnage
data_list = []
for i in range(n_rows):
    # Génération d'une date unique en ajoutant des jours
    date = start_date + timedelta(days=i % 365)

    row = {
        "indicator": np.random.choice(indicators),
        "country": np.random.choice(countries),
        "kind": np.random.choice(kinds),
        "model": np.random.choice(models),
        "training": np.random.choice(trainings),
        "week": np.random.choice(weeks),
        "horizon": np.random.choice(horizons),
        "date": date,
        "value": np.random.uniform(10, 100),
        "lower_bound": None if np.random.random() > 0.7 else np.random.uniform(5, 50),
        "upper_bound": None if np.random.random() > 0.7 else np.random.uniform(50, 150),
        "quality_score": np.random.uniform(0, 1),
        "notes": np.random.choice(
            ["OK", "Warning", None, "Error"], p=[0.7, 0.15, 0.1, 0.05]
        ),
    }
    data_list.append(row)

# Création du DataFrame
df_origin = pd.DataFrame(data_list)

# Suppression des doublons potentiels sur la clé primaire composite
pk_columns = [
    "indicator",
    "country",
    "kind",
    "model",
    "training",
    "week",
    "horizon",
    "date",
]
df_origin = df_origin.drop_duplicates(subset=pk_columns, keep="first")

# Conversion en datetime
df_origin["date"] = pd.to_datetime(df_origin["date"])

# Définition des labels pour les colonnes
labels = {
    "indicator": "Indicateur",
    "country": "Pays",
    "kind": "Type",
    "model": "Modèle",
    "training": "Entraînement",
    "week": "Semaine",
    "horizon": "Horizon",
    "date": "Date",
    "value": "Valeur",
    "lower_bound": "Borne inférieure",
    "upper_bound": "Borne supérieure",
    "quality_score": "Score de qualité",
    "notes": "Notes",
}

# Définition des paramètres explicites
CATEGORICAL_THRESHOLD = 10
CATALOG_PATH_NO_PK = os.path.join("../outputs", "db_no_pk.ducklake")
DATA_PATH_NO_PK = os.path.join("../outputs", "db_no_pk_data/")
CATALOG_PATH_PK = os.path.join("../outputs", "db_with_pk.ducklake")
DATA_PATH_PK = os.path.join("../outputs", "db_with_pk_data/")

# Affichage
print(f"Nombre de lignes générées : {len(df_origin)}")
print(f"Categorical threshold : {CATEGORICAL_THRESHOLD}")
print(f"Catalogue (sans PK) : {CATALOG_PATH_NO_PK}")
print("\nAperçu des données :")
df_origin.head()

## 2. Construction du schéma <a id="section_2"></a>

### 2.1. Initialisation de la classe <a id="section_2_1"></a>

In [ ]:
# Initialisation du schéma
schema_builder = SchemaBuilder(
    df=df_origin, categorical_threshold=CATEGORICAL_THRESHOLD
)

### 2.2 Construction des méta-données <a id="section_2_2"></a>

In [ ]:
# Construction du jeu de métadonnées
df_metadata = schema_builder.create_metadata_table(column_labels=labels)

df_metadata.head()

### 2.3. Construction des tables de dimensions <a id="section_2_3"></a>

In [ ]:
# Construction des types de dimensions
dimension_tables = schema_builder.create_dimension_tables(column_labels=labels)
dimension_tables["indicator"].head()

### 2.4. Construction de la table d'information <a id="section_2_4"></a>

In [ ]:
# Construction de la table d'informations
df_fact = schema_builder.create_fact_table(column_labels=labels)
df_fact.head()

### 2.5. Création de l'ensemble des tables <a id="section_2_5"></a>

In [ ]:
# Création de l'ensemble des tables du schéma
df_metadata, dimension_tables, df_fact = schema_builder.build(column_labels=labels)

### 2.6. Compatibilité multi-backend — pandas, polars et narwhals <a id="section_2_6"></a>

`SchemaBuilder` accepte n'importe quel DataFrame compatible narwhals grâce au paramètre typé `IntoDataFrame`.
Les exemples précédents passaient un DataFrame **pandas** ; on peut passer exactement les mêmes données sous forme de DataFrame **polars** et obtenir un résultat identique.

Les tables retournées (`df_metadata`, `dimension_tables`, `df_fact`) sont des `nw.DataFrame`.  
Pour récupérer le DataFrame dans un format natif spécifique, utiliser `nw.to_native()`.

In [ ]:
# --- Entrée pandas (déjà utilisée dans les sections précédentes) ---
schema_pandas = SchemaBuilder(df=df_origin, categorical_threshold=CATEGORICAL_THRESHOLD)
meta_pandas, dims_pandas, fact_pandas = schema_pandas.build(column_labels=labels)
print(f"Type de l'entrée  : {type(df_origin).__name__}")
print(f"Type de la sortie : {type(meta_pandas).__name__}")

# --- Entrée polars (même données) ---
df_polars = pl.from_pandas(df_origin)
schema_polars = SchemaBuilder(df=df_polars, categorical_threshold=CATEGORICAL_THRESHOLD)
meta_polars, dims_polars, fact_polars = schema_polars.build(column_labels=labels)
print(f"\nType de l'entrée  : {type(df_polars).__name__}")
print(f"Type de la sortie : {type(meta_polars).__name__}")

# --- Vérification : les deux sorties sont identiques ---
assert meta_pandas["name"].to_list() == meta_polars["name"].to_list(), (
    "Les métadonnées diffèrent !"
)
print("\nLes sorties pandas et polars sont identiques.")

# --- Conversion du résultat vers un format natif spécifique ---
# nw.to_native() retourne le DataFrame dans le backend d'origine (ici polars, backend
# interne)
fact_native = nw.to_native(fact_polars)
print(f"\nConversion nw.to_native() → {type(fact_native).__name__}")
fact_native.head(3)

## 3. Construction de la base de données <a id="section_3"></a>

### 3.1. Construction d'une base de données sans clé primaire <a id="section_3_1"></a>

#### 3.1.1. Initialisation du builder <a id="section_3_1_1"></a>

In [ ]:
# Suppression du catalogue et des données existants pour garantir un état initial propre
for suffix in ["", ".wal"]:
    path_to_remove = CATALOG_PATH_NO_PK + suffix
    if os.path.exists(path_to_remove):
        os.remove(path_to_remove)
        print(f"Fichier supprimé : {path_to_remove}")
if os.path.exists(DATA_PATH_NO_PK):
    shutil.rmtree(DATA_PATH_NO_PK)
    print(f"Répertoire supprimé : {DATA_PATH_NO_PK}")

# Création de la connexion DuckLake
conn_no_pk = DuckLakeConnector(CATALOG_PATH_NO_PK, DATA_PATH_NO_PK).connect()

# Initialisation du builder
builder = DuckLakeTablesBuilder(
    df=df_origin,
    categorical_threshold=CATEGORICAL_THRESHOLD,
    connection=conn_no_pk,
)

#### 3.1.2. Construction du schéma <a id="section_3_1_2"></a>

In [ ]:
# Construction du schéma duckDB
builder.build_schema()

#### 3.1.3. Affichage du schéma <a id="section_3_1_3"></a>

In [ ]:
# Affichage du schéma
builder.display_schema()

#### 3.1.4. Exemple de requête <a id="section_3_1_4"></a>

In [ ]:
# Requête de la table d'information
print(builder.conn.execute("SELECT * FROM dim_model").fetchall())

### 3.2. Construction d'une base de données avec clé primaire <a id="section_3_2"></a>

#### 3.2.1. Création d'une clé primaire composite <a id="section_3_2_1"></a>

In [ ]:
# Définition des clés primaires composites
# Clés adaptées aux données synthétiques (pas de NaN, combinaison unique)
composite_pk = [
    "indicator",
    "country",
    "kind",
    "model",
    "training",
    "week",
    "horizon",
    "date",
]

# Vérification que les colonnes de clé primaire n'ont pas de NaN
print("Vérification de l'absence de NaN dans les colonnes de clé primaire :")
for col in composite_pk:
    nan_count = df_origin[col].isna().sum()
    print(f"  - {col}: {nan_count} NaN")

# Vérification de l'unicité de la combinaison
duplicates = df_origin.duplicated(subset=composite_pk).sum()
print(f"\nNombre de combinaisons dupliquées : {duplicates}")

# Suppression du catalogue et des données existants pour garantir un état initial propre
for suffix in ["", ".wal"]:
    path_to_remove = CATALOG_PATH_PK + suffix
    if os.path.exists(path_to_remove):
        os.remove(path_to_remove)
        print(f"Fichier supprimé : {path_to_remove}")
if os.path.exists(DATA_PATH_PK):
    shutil.rmtree(DATA_PATH_PK)
    print(f"Répertoire supprimé : {DATA_PATH_PK}")

# Création de la connexion DuckLake avec clé primaire
conn_pk = DuckLakeConnector(CATALOG_PATH_PK, DATA_PATH_PK).connect()

# Initialisation du builder avec clés primaires
builder_with_pk = DuckLakeTablesBuilder(
    df=df_origin,
    categorical_threshold=CATEGORICAL_THRESHOLD,
    primary_keys=composite_pk,
    connection=conn_pk,
)

# Construction du schéma
builder_with_pk.build_schema()

#### 3.2.2 Condition d'unicité sur la clé primaire lors de la création de la table <a id="section_3_2_2"></a>

DuckLake ne supporte pas les contraintes DDL `PRIMARY KEY`. L'unicité est gérée applicativement dans `build_schema()` via le paramètre `check_duplicates=True` (valeur par défaut). Avec `keep='first'` ou `keep='last'`, une occurrence est conservée par groupe de doublons. Avec `keep=False`, toutes les occurrences dupliquées sont supprimées.

In [ ]:
# Création d'un DataFrame avec doublons intentionnels
df_with_duplicates = pd.concat(
    [df_origin.head(10), df_origin.head(5)], ignore_index=True
)
print(
    f"DataFrame avec {len(df_with_duplicates)} lignes (dont 5 doublons sur la clé"
    f" primaire)"
)

# DuckLake ne supportant pas les contraintes DDL, l'unicité est gérée applicativement.
# build_schema(check_duplicates=True, keep='first') supprime silencieusement les
# doublons.
builder_dup = DuckLakeTablesBuilder(
    df=df_with_duplicates,
    categorical_threshold=CATEGORICAL_THRESHOLD,
    primary_keys=composite_pk,
)
builder_dup.build_schema(check_duplicates=True, keep="first")
n_rows = builder_dup.conn.execute("SELECT COUNT(*) FROM fact_table").fetchone()[0]
print(f"✓ Doublons traités applicativement : {n_rows} lignes dans fact_table")
print(
    f"  ({len(df_with_duplicates) - n_rows} doublon(s) supprimé(s) avec keep='first')"
)

## 4. categorical_threshold=None — pas de tables de dimension <a id="section_4"></a>

Deux comportements opposés sont illustrés :
- `categorical_threshold=None` : aucune table de dimension n'est créée ; les colonnes catégorielles sont stockées directement en VARCHAR dans la fact table.
- `categorical_threshold=10` : les colonnes avec ≤ 10 valeurs uniques génèrent une table de dimension.

In [ ]:
# --- Cas 1 : categorical_threshold=None ---
# Aucune table de dimension n'est créée
builder_no_dim = DuckLakeTablesBuilder(
    df=df_origin,
    categorical_threshold=None,
    primary_keys=composite_pk,
)
builder_no_dim.build_schema()

# Vérification des métadonnées : is_categorical doit valoir False partout
df_meta_no_dim = builder_no_dim.conn.execute("SELECT * FROM metadata").fetchdf()
print("=== categorical_threshold=None ===")
print(f"Colonnes catégorielles : {df_meta_no_dim['is_categorical'].sum()}")
print(df_meta_no_dim[["name", "python_type", "is_categorical"]])

# Aucune table de dimension
all_tables = [row[0] for row in builder_no_dim.conn.execute("SHOW TABLES").fetchall()]
dim_tables = [t for t in all_tables if t.startswith("dim_")]
print(f"\nTables de dimension créées : {dim_tables}")
print(f"Tables présentes : {all_tables}")

In [ ]:
# --- Cas 2 : categorical_threshold=10 (comportement explicite) ---
# Les colonnes avec ≤ 10 valeurs uniques génèrent une table de dimension
builder_with_dim = DuckLakeTablesBuilder(
    df=df_origin,
    categorical_threshold=CATEGORICAL_THRESHOLD,  # = 10, explicite
    primary_keys=composite_pk,
)
builder_with_dim.build_schema()

df_meta_with_dim = builder_with_dim.conn.execute("SELECT * FROM metadata").fetchdf()
print("=== categorical_threshold=10 ===")
print(f"Colonnes catégorielles : {df_meta_with_dim['is_categorical'].sum()}")
print(
    df_meta_with_dim.loc[df_meta_with_dim["is_categorical"], ["name", "is_categorical"]]
)

all_tables_with_dim = [
    row[0] for row in builder_with_dim.conn.execute("SHOW TABLES").fetchall()
]
dim_tables_with_dim = [t for t in all_tables_with_dim if t.startswith("dim_")]
print(f"\nTables de dimension créées : {dim_tables_with_dim}")